In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Load all cleaned datasets
daily_activity = pd.read_csv(r'C:\Users\SILVERWING\Desktop\other\Data Analysis\Case Study\bellabeat_case_study\02_data_processed\dailyActivity_combined.csv', 
                          thousands=',')
sleep_day = pd.read_csv(r'C:\Users\SILVERWING\Desktop\other\Data Analysis\Case Study\bellabeat_case_study\02_data_processed\sleepDay_merged.csv', 
                          thousands=',')
hourly_steps = pd.read_csv(r'C:\Users\SILVERWING\Desktop\other\Data Analysis\Case Study\bellabeat_case_study\02_data_processed\hourlySteps_combined.csv', 
                          thousands=',')
hourly_intensities = pd.read_csv(r'C:\Users\SILVERWING\Desktop\other\Data Analysis\Case Study\bellabeat_case_study\02_data_processed\hourlyIntensities_combined.csv', 
                          thousands=',')
hourly_calories = pd.read_csv(r'C:\Users\SILVERWING\Desktop\other\Data Analysis\Case Study\bellabeat_case_study\02_data_processed\hourlyCalories_combined.csv', 
                          thousands=',')
weight_log = pd.read_csv(r'C:\Users\SILVERWING\Desktop\other\Data Analysis\Case Study\bellabeat_case_study\02_data_processed\weightLogInfo_combined.csv', 
                          thousands=',')

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
# Preview each dataset
print("=== DAILY ACTIVITY ===")
print(f"Shape: {daily_activity.shape}")
print(daily_activity.head(3))

print("\n=== SLEEP DAY ===")
print(f"Shape: {sleep_day.shape}")
print(sleep_day.head(3))

print("\n=== WEIGHT LOG ===")
print(f"Shape: {weight_log.shape}")
print(weight_log.head(3))

=== DAILY ACTIVITY ===
Shape: (1382, 17)
           Id ActivityDate  TotalSteps  TotalDistance  TrackerDistance  \
0  1503960366    3/25/2016     11004.0           7.11             7.11   
1  1503960366    3/26/2016     17609.0          11.55            11.55   
2  1503960366    3/27/2016     12736.0           8.53             8.53   

   LoggedActivitiesDistance  VeryActiveDistance  ModeratelyActiveDistance  \
0                       0.0                2.57                      0.46   
1                       0.0                6.92                      0.73   
2                       0.0                4.66                      0.16   

   LightActiveDistance  SedentaryActiveDistance  VeryActiveMinutes  \
0                 4.07                      0.0               33.0   
1                 3.91                      0.0               89.0   
2                 3.71                      0.0               56.0   

   FairlyActiveMinutes  LightlyActiveMinutes  SedentaryMinutes  Calories

In [4]:
# Fix date formatting
daily_activity['ActivityDate'] = pd.to_datetime(daily_activity['ActivityDate'])
sleep_day['SleepDay'] = pd.to_datetime(sleep_day['SleepDay'])
weight_log['Date'] = pd.to_datetime(weight_log['Date']).dt.date

# Fix hourly datetime columns
hourly_steps['ActivityHour'] = pd.to_datetime(hourly_steps['ActivityHour'])
hourly_intensities['ActivityHour'] = pd.to_datetime(hourly_intensities['ActivityHour'])
hourly_calories['ActivityHour'] = pd.to_datetime(hourly_calories['ActivityHour'])

print("All dates formatted successfully!")

All dates formatted successfully!


In [5]:
# Summary statistics for daily activity
print("=== DAILY ACTIVITY SUMMARY ===")
print(daily_activity[['TotalSteps', 'Calories', 'VeryActiveMinutes', 
                        'FairlyActiveMinutes', 'LightlyActiveMinutes', 
                        'SedentaryMinutes']].describe().round(2))

print("\n=== SLEEP SUMMARY ===")
print(sleep_day[['TotalMinutesAsleep', 'TotalTimeInBed', 
                  'TimeSpentAwakeInBed']].describe().round(2))

print("\n=== WEIGHT SUMMARY ===")
print(weight_log[['WeightKg', 'BMI']].describe().round(2))

=== DAILY ACTIVITY SUMMARY ===
       TotalSteps  Calories  VeryActiveMinutes  FairlyActiveMinutes  \
count     1382.00   1382.00            1382.00              1382.00   
mean      7358.87   2289.82              19.89                13.55   
std       5188.21    721.89              31.78                26.51   
min          0.00    120.00               0.00                 0.00   
25%       3282.25   1816.75               0.00                 0.00   
50%       7123.00   2122.00               2.00                 6.00   
75%      10612.50   2777.25              30.00                18.00   
max      36019.00   4900.00             210.00               660.00   

       LightlyActiveMinutes  SedentaryMinutes  
count               1382.00           1382.00  
mean                 187.36            993.81  
std                  113.06            306.39  
min                    0.00             61.00  
25%                  114.25            730.00  
50%                  195.00           105

In [6]:
# Merge activity and sleep data
activity_sleep = pd.merge(daily_activity, sleep_day, 
                           left_on=['Id', 'ActivityDate'],
                           right_on=['Id', 'SleepDay'],
                           how='inner')

# Calculate correlations
correlations = activity_sleep[['TotalSteps', 'Calories', 
                                'VeryActiveMinutes',
                                'SedentaryMinutes',
                                'TotalMinutesAsleep',
                                'TimeSpentAwakeInBed']].corr().round(2)

print("=== CORRELATION MATRIX ===")
print(correlations)

=== CORRELATION MATRIX ===
                     TotalSteps  Calories  VeryActiveMinutes  \
TotalSteps                 1.00      0.43               0.54   
Calories                   0.43      1.00               0.60   
VeryActiveMinutes          0.54      0.60               1.00   
SedentaryMinutes          -0.07      0.16              -0.01   
TotalMinutesAsleep        -0.20     -0.05              -0.10   
TimeSpentAwakeInBed        0.02     -0.29              -0.08   

                     SedentaryMinutes  TotalMinutesAsleep  TimeSpentAwakeInBed  
TotalSteps                      -0.07               -0.20                 0.02  
Calories                         0.16               -0.05                -0.29  
VeryActiveMinutes               -0.01               -0.10                -0.08  
SedentaryMinutes                 1.00               -0.59                -0.17  
TotalMinutesAsleep              -0.59                1.00                -0.01  
TimeSpentAwakeInBed             -0.17 

In [7]:
# Create summary tables for Tableau

# 1. Activity segments summary
user_avg = daily_activity.groupby('Id').agg(
    avg_steps=('TotalSteps', 'mean'),
    avg_calories=('Calories', 'mean'),
    avg_sedentary_mins=('SedentaryMinutes', 'mean'),
    avg_very_active_mins=('VeryActiveMinutes', 'mean')
).reset_index()

# Add activity segment labels
def categorize_activity(steps):
    if steps >= 10000:
        return 'Very Active'
    elif steps >= 7500:
        return 'Moderately Active'
    elif steps >= 5000:
        return 'Lightly Active'
    else:
        return 'Sedentary'

user_avg['ActivitySegment'] = user_avg['avg_steps'].apply(categorize_activity)

# 2. Hourly steps summary
hourly_steps['Hour'] = hourly_steps['ActivityHour'].dt.hour
hourly_summary = hourly_steps.groupby('Hour').agg(
    avg_steps=('StepTotal', 'mean'),
    max_steps=('StepTotal', 'max')
).reset_index()

# 3. Day of week summary
daily_activity['DayOfWeek'] = daily_activity['ActivityDate'].dt.day_name()
day_summary = daily_activity.groupby('DayOfWeek').agg(
    avg_steps=('TotalSteps', 'mean'),
    avg_calories=('Calories', 'mean'),
    avg_sedentary=('SedentaryMinutes', 'mean')
).reset_index()

# 4. Sleep summary per user
sleep_summary = sleep_day.groupby('Id').agg(
    avg_hours_asleep=('TotalMinutesAsleep', 'mean'),
    avg_time_in_bed=('TotalTimeInBed', 'mean'),
    avg_awake_in_bed=('TimeSpentAwakeInBed', 'mean')
).reset_index()
sleep_summary['avg_hours_asleep'] = (sleep_summary['avg_hours_asleep'] / 60).round(2)
sleep_summary['avg_time_in_bed'] = (sleep_summary['avg_time_in_bed'] / 60).round(2)

# Export all to CSV for Tableau
user_avg.to_csv('tableau_activity_segments.csv', index=False)
hourly_summary.to_csv('tableau_hourly_steps.csv', index=False)
day_summary.to_csv('tableau_day_of_week.csv', index=False)
sleep_summary.to_csv('tableau_sleep_summary.csv', index=False)

print("All Tableau files exported successfully!")
print(f"\nActivity Segments shape: {user_avg.shape}")
print(f"Hourly Summary shape: {hourly_summary.shape}")
print(f"Day of Week shape: {day_summary.shape}")
print(f"Sleep Summary shape: {sleep_summary.shape}")

All Tableau files exported successfully!

Activity Segments shape: (35, 6)
Hourly Summary shape: (24, 3)
Day of Week shape: (7, 4)
Sleep Summary shape: (24, 4)
